# RAG Augmentation — v4

**Changes over v3:**
- Rate limits correctly tuned to each backend's free tier:
  - **Gemini 2.5 Flash** (default): 10 RPM → 7 s sleep per call, 250 RPD → ~62 pairs/day
  - **Groq / Llama-3.3-70B** (alternative): 30 RPM → 3 s sleep per call, 1 000 RPD → ~250 pairs/day
- Single `BACKEND` switch controls everything — prompts, client, sleep, and model string.
- Both clients share the same `call_api()` interface so no downstream code changes when switching.
- Daily capacity estimate printed at startup so you can plan multi-day runs in advance.

**Four scenarios per pair (unchanged):**

| Thesis Label | Scenario | NLI Label |
|---|---|---|
| Skenario A | `entailment` | entailment |
| Skenario B | `contradiction` | contradiction |
| Skenario C | `neutral` | neutral |
| Skenario D | `fake_drug` | neutral (OOV-filtered) |

In [10]:
import os
import time
import glob
import math
import xml.etree.ElementTree as ET

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [11]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                     BACKEND SWITCH                              ║
# ║  Set BACKEND = "gemini" or "groq", then fill in the API key.   ║
# ╚══════════════════════════════════════════════════════════════════╝
BACKEND = "groq"   # <── change to "groq" to use Llama-3 instead

# ── Per-backend settings ─────────────────────────────────────────────────────
BACKEND_CONFIG = {
    "gemini": {
        # Gemini 2.5 Flash free tier: 10 RPM, 250 RPD
        # 60 s / 10 RPM = 6 s minimum between calls.
        # Using 7 s adds a small buffer against clock skew / burst rejection.
        "api_key":    os.getenv("GEMINI_API_KEY"),
        "model":      "gemini-2.5-flash",
        "sleep_sec":  7,
        "rpm":        10,
        "rpd":        250,
    },
    "groq": {
        # Groq free tier: 30 RPM, 1 000 RPD (for llama-3.1-8b-instant)
        # 60 s / 30 RPM = 2 s minimum. Using 3 s for safety.
        # TPM cap is 6 000 — keep prompts concise to avoid hitting it.
        "api_key":   os.getenv("GROQ_API_KEY"),
        "model":     "llama-3.1-8b-instant", 
        "sleep_sec":  3,
        "rpm":        30,
        "rpd":        1_000,
    },
}

cfg        = BACKEND_CONFIG[BACKEND]
API_KEY    = cfg["api_key"]
MODEL_NAME = cfg["model"]
SLEEP_SEC  = cfg["sleep_sec"]

# ── Shared settings ──────────────────────────────────────────────────────────

# Windows
# DDI_TEST_ROOT = r"DDICorpus\Test\Test for DDI Extraction task"

# # Mac
DDI_TEST_ROOT = "DDICorpus/Test/Test for DDI Extraction task"

XML_DIRS = [
    os.path.join(DDI_TEST_ROOT, "DrugBank"),
    os.path.join(DDI_TEST_ROOT, "MedLine"),
]

OUTPUT_CSV       = f"synthetic_rag_dataset_{BACKEND}.csv"
CHECKPOINT_CSV   = f"synthetic_rag_checkpoint_{BACKEND}.csv"
CHECKPOINT_EVERY = 20
MAX_RETRIES      = 4
FAKE_DRUG_NAME   = "Synthetozole"
SCENARIOS        = ["entailment", "contradiction", "neutral", "fake_drug"]

# ── Capacity estimate ────────────────────────────────────────────────────────
# Each pair requires 4 API calls (one per scenario).
calls_per_pair       = len(SCENARIOS)
pairs_per_day        = cfg["rpd"] // calls_per_pair
wall_time_per_pair_s = calls_per_pair * SLEEP_SEC

print(f"Backend      : {BACKEND.upper()} / {MODEL_NAME}")
print(f"Rate limit   : {cfg['rpm']} RPM  |  {cfg['rpd']} RPD")
print(f"Sleep/call   : {SLEEP_SEC} s")
print(f"Wall time    : ~{wall_time_per_pair_s} s per pair")
print(f"Daily budget : ~{pairs_per_day} pairs/day  ({pairs_per_day * calls_per_pair} API calls)")
print(f"Output CSV   : {OUTPUT_CSV}")

Backend      : GROQ / llama-3.1-8b-instant
Rate limit   : 30 RPM  |  1000 RPD
Sleep/call   : 3 s
Wall time    : ~12 s per pair
Daily budget : ~250 pairs/day  (1000 API calls)
Output CSV   : synthetic_rag_dataset_groq.csv


In [12]:
# ── Initialise the chosen client ─────────────────────────────────────────────
if BACKEND == "gemini":
    from google import genai
    from google.genai import types as genai_types
    gemini_client = genai.Client(api_key=API_KEY)
    print("Gemini client ready.")

elif BACKEND == "groq":
    from groq import Groq
    groq_client = Groq(api_key=API_KEY)
    print("Groq client ready.")

Groq client ready.


In [13]:
# ── DDI-2013 XML Parser ───────────────────────────────────────────────────────
def parse_ddi_xml_file(filepath: str) -> list[dict]:
    rows   = []
    source = "DrugBank" if "DrugBank" in filepath else "MedLine"
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"  [WARN] XML parse error in {filepath}: {e}")
        return rows

    doc_id = root.attrib.get("id", os.path.basename(filepath))

    for sent in root.findall("sentence"):
        sent_id   = sent.attrib["id"]
        sent_text = sent.attrib.get("text", "").strip()
        if not sent_text:
            continue

        entities = {
            e.attrib["id"]: e.attrib.get("text", "").strip()
            for e in sent.findall("entity")
        }

        for pair in sent.findall("pair"):
            ddi_flag = pair.attrib.get("ddi", "false").lower() == "true"
            itype    = pair.attrib.get("type", "none") if ddi_flag else "none"
            e1_id    = pair.attrib["e1"]
            e2_id    = pair.attrib["e2"]
            e1_text  = entities.get(e1_id, "")
            e2_text  = entities.get(e2_id, "")
            if not e1_text or not e2_text:
                continue
            rows.append({
                "id":               pair.attrib["id"],
                "document_id":      doc_id,
                "sentence_id":      sent_id,
                "source":           source,
                "sentence":         sent_text,
                "e1_id":            e1_id,
                "e2_id":            e2_id,
                "e1_text":          e1_text,
                "e2_text":          e2_text,
                "ddi":              ddi_flag,
                "interaction_type": itype,
            })
    return rows


def parse_ddi_corpus(xml_dirs: list[str]) -> pd.DataFrame:
    all_rows = []
    for directory in xml_dirs:
        xml_files = sorted(glob.glob(os.path.join(directory, "*.xml")))
        print(f"  {directory}: {len(xml_files)} files")
        for fp in xml_files:
            all_rows.extend(parse_ddi_xml_file(fp))
    return pd.DataFrame(all_rows)


print("XML parser defined.")

XML parser defined.


In [14]:
print("Parsing DDI-2013 test XML files...")
df_test = parse_ddi_corpus(XML_DIRS)

print(f"\nTotal pairs        : {len(df_test):,}")
print(f"Unique sentences   : {df_test['sentence_id'].nunique():,}")
print(f"Unique documents   : {df_test['document_id'].nunique():,}")
print()
print("Source breakdown:")
print(df_test["source"].value_counts().to_string())
print()
print("Interaction type distribution:")
print(df_test["interaction_type"].value_counts().to_string())
print()

# Warn if total calls exceed today's daily budget
total_calls = len(df_test) * calls_per_pair
days_needed = math.ceil(total_calls / cfg["rpd"])
print(f"Total API calls needed : {total_calls:,}")
if days_needed > 1:
    print(f"[INFO] Exceeds 1-day RPD budget. Estimated run time: {days_needed} day(s).")
    print("       Checkpoint saves every 20 pairs — safe to stop and resume.")

Parsing DDI-2013 test XML files...
  DDICorpus/Test/Test for DDI Extraction task/DrugBank: 158 files
  DDICorpus/Test/Test for DDI Extraction task/MedLine: 33 files

Total pairs        : 5,716
Unique sentences   : 790
Unique documents   : 191

Source breakdown:
source
DrugBank    5265
MedLine      451

Interaction type distribution:
interaction_type
none         4737
effect        360
mechanism     302
advise        221
int            96

Total API calls needed : 22,864
[INFO] Exceeds 1-day RPD budget. Estimated run time: 23 day(s).
       Checkpoint saves every 20 pairs — safe to stop and resume.


In [15]:
# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a clinical pharmacology assistant generating concise summaries "
    "of drug-drug interaction (DDI) information for a medical literature review system. "
    "Write in 5-6 sentences. Do not add disclaimers, safety warnings, or meta-commentary "
    "about the nature of the task. Output the summary only."
)


def build_prompts(context: str, e1: str, e2: str) -> dict:
    return {
        # Skenario A — faithful summary
        "entailment": (
            f"Summarise the following drug-drug interaction finding accurately and faithfully. "
            f"Preserve the clinical direction of the effect (e.g. increases, decreases, "
            f"no significant change).\n\nContext: {context}"
        ),
        # Skenario B — clinical inversion (avoids safety-filter triggers)
        "contradiction": (
            f"You are writing a summary that reflects an alternative clinical finding "
            f"that directly contradicts the source text. "
            f"If the source states no significant interaction exists between {e1} and {e2}, "
            f"your summary must state that a clinically significant interaction does exist. "
            f"If the source states an interaction exists (e.g. one drug increases or decreases "
            f"the levels of the other), your summary must state the opposite effect or that "
            f"no interaction occurs. Keep both drug names ({e1} and {e2}) in the summary. "
            f"Do not add hedging phrases like 'however' or 'contrary to some reports'. "
            f"State the contradicting finding as clinical fact.\n\nContext: {context}"
        ),
        # Skenario C — irrelevant, hard exclusion on interaction content
        "neutral": (
            f"Write a brief description of {e1} and {e2} that mentions both drugs by name "
            f"but discusses ONLY their pharmacological drug class or general mechanism of action. "
            f"Do NOT mention, imply, or reference any interaction, combined effect, "
            f"pharmacokinetic change, clinical outcome, or safety profile between them. "
            f"The output must be clinically irrelevant to whether these drugs interact.\n\nContext: {context}"
        ),
        # Skenario D — OOV guardrail trigger
        "fake_drug": (
            f"Summarise the following drug-drug interaction finding accurately, "
            f"but replace every mention of '{e2}' with the drug name '{FAKE_DRUG_NAME}'. "
            f"Keep all other clinical details and drug names ({e1}) unchanged. "
            f"Do not add any explanation of the substitution.\n\nContext: {context}"
        ),
    }


print("Prompt builder defined.")

Prompt builder defined.


In [16]:
# ── Unified call_api() — dispatches to Gemini or Groq transparently ──────────
def call_api(prompt: str, retries: int = MAX_RETRIES) -> str:
    """
    Send a prompt to the configured backend and return the response text.
    Retries with exponential backoff on any exception.
    Sleeps SLEEP_SEC after every successful call to respect the free-tier RPM.
    """
    for attempt in range(retries):
        try:
            if BACKEND == "gemini":
                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=genai_types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0.7,
                        max_output_tokens=512,
                    ),
                )
                text = response.text.strip()

            elif BACKEND == "groq":
                # Groq follows the OpenAI chat-completion format.
                # System prompt goes in the messages array as role="system".
                response = groq_client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    temperature=0.7,
                    max_tokens=512,
                )
                text = response.choices[0].message.content.strip()

            time.sleep(SLEEP_SEC)
            return text

        except Exception as e:
            wait = SLEEP_SEC * (2 ** attempt)
            print(f"    [Attempt {attempt+1}/{retries}] {type(e).__name__}: {e} — retry in {wait}s")
            time.sleep(wait)

    print(f"    [FAILED] All {retries} attempts exhausted.")
    return ""


def validate_response(text: str) -> str:
    """Flag empty, refused, or suspiciously short generations."""
    if not text:
        return "empty"
    refusal_markers = [
        "i cannot", "i'm unable", "i am unable",
        "as an ai", "i can't assist", "i can't provide",
        "this request", "harmful content",
    ]
    if any(m in text.lower() for m in refusal_markers):
        return "refused"
    if len(text.split()) < 15:
        return "too_short"
    return "ok"


print("API helpers defined.")

API helpers defined.


In [ ]:


# Resume from checkpoint if one exists
if os.path.exists(CHECKPOINT_CSV):
    df_existing    = pd.read_csv(CHECKPOINT_CSV)
    done_ids       = set(df_existing["original_id"].unique())
    augmented_data = df_existing.to_dict("records")
    print(f"Resuming — {len(done_ids)} pairs already done ({len(augmented_data)} records).")
else:
    done_ids       = set()
    augmented_data = []
    print("No checkpoint — starting fresh.")

rows_processed = 0

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Pairs"):
    pair_id = row["id"]
    if pair_id in done_ids:
        continue

    context = row["sentence"]
    e1      = row["e1_text"]
    e2      = row["e2_text"]
    prompts = build_prompts(context, e1, e2)

    for scenario in SCENARIOS:
        synthetic_text = call_api(prompts[scenario])
        status         = validate_response(synthetic_text)

        if status != "ok":
            print(f"  [WARN] {pair_id} | {scenario} | {status}")

        augmented_data.append({
            "original_id":          pair_id,
            "source":               row["source"],
            "e1_text":              e1,
            "e2_text":              e2,
            "premise":              context,
            "synthetic_rag_output": synthetic_text,
            "interaction_type":     row["interaction_type"],
            "scenario":             scenario,
            "model":                MODEL_NAME,
        })

    done_ids.add(pair_id)
    rows_processed += 1

    if rows_processed % CHECKPOINT_EVERY == 0:
        pd.DataFrame(augmented_data).to_csv(CHECKPOINT_CSV, index=False)
        print(f"  [Checkpoint] {len(augmented_data)} records after {rows_processed} pairs.")

print(f"\nDone. Total records: {len(augmented_data):,}")

Resuming — 1580 pairs already done (6320 records).


Pairs:   0%|          | 0/5716 [00:00<?, ?it/s]

    [Attempt 1/4] RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01krrfs8fxep98ax78zaw3xxnh` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499963, Requested 412. Please try again in 1m4.8s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}} — retry in 3s


KeyboardInterrupt: 

In [ ]:
# ── Save and quality report ───────────────────────────────────────────────────
df_augmented = pd.DataFrame(augmented_data)
df_augmented.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_augmented):,} rows to {OUTPUT_CSV}")
print()
print("Scenario × Source distribution:")
print(df_augmented.groupby(["source", "scenario"]).size().unstack(fill_value=0).to_string())
# print(df_augmented.groupby(["scenario", "response_status"]).size().to_string())

# flagged = df_augmented[df_augmented["response_status"] != "ok"]
# if len(flagged):
#     print(f"\n{len(flagged)} flagged responses need review:")
#     print(flagged[["original_id", "scenario", "response_status"]].to_string())

# if os.path.exists(CHECKPOINT_CSV):
#     os.remove(CHECKPOINT_CSV)
#     print("\nCheckpoint removed.")

Saved 6,323 rows to synthetic_rag_dataset_groq.csv

Scenario × Source distribution:
scenario  contradiction  entailment  fake_drug  neutral
source                                                 
DrugBank           1581        1581       1580     1581

Response status breakdown:


In [28]:
# ── Spot-check: all 4 scenarios for one pair ─────────────────────────────────
sample_id = df_augmented["original_id"].iloc[0]
sample    = df_augmented[df_augmented["original_id"] == sample_id]

print(f"=== Source (id={sample_id}) ===")
print("Premise       :", sample.iloc[0]["premise"])
print("e1 / e2       :", sample.iloc[0]["e1_text"], "/", sample.iloc[0]["e2_text"])
print("Interaction   :", sample.iloc[0]["interaction_type"])


=== Source (id=DDI-DrugBank.d610.s0.p0) ===
Premise       : Pharmacokinetic properties of abacavir were not altered by the addition of either lamivudine or zidovudine or the combination of lamivudine and zidovudine.
e1 / e2       : abacavir / lamivudine
Interaction   : none
